In [1]:
# =========================
# INSTALL RDKIT FROM KAGGLE INPUT WHEEL
# =========================

import os
import glob

rdkit_wheels = glob.glob("/kaggle/input/**/*.whl", recursive=True)
rdkit_wheels = [p for p in rdkit_wheels if "rdkit" in p.lower()]

print("Found RDKit wheels:")
for p in rdkit_wheels:
    print(p)

if len(rdkit_wheels) == 0:
    raise FileNotFoundError("No RDKit wheel found. Please add an RDKit wheel dataset to Kaggle inputs.")

wheel_path = rdkit_wheels[0]
print("Installing:", wheel_path)

!pip install --no-index --no-deps "$wheel_path"

Found RDKit wheels:
/kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl
Installing: /kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl


In [2]:
import os
import glob
import random
import copy
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import KFold
from tqdm import tqdm

SEED = 42
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

N_SPLITS = 5
BATCH_SIZE = 32
HIDDEN_DIM = 64
LR = 1e-3
NUM_EPOCHS = 20

OUTPUT_DIR = "./gcn_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [4]:
def find_data_dir():
    found = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    for path in found:
        d = os.path.dirname(path)
        if os.path.exists(os.path.join(d, "test.csv")):
            return d

    for d in ["../data", "./data"]:
        if os.path.exists(os.path.join(d, "train.csv")):
            return d

    raise FileNotFoundError("Could not find train.csv")

DATA_DIR = find_data_dir()
SUPP_DIR = os.path.join(DATA_DIR, "train_supplement")

print("DATA_DIR:", DATA_DIR)

DATA_DIR: /kaggle/input/competitions/neurips-open-polymer-prediction-2025


In [5]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

print("Train shape:", train_df.shape)
print("\nMissing ratio:")
print((train_df[TARGETS].isna().mean() * 100).round(1))

Train shape: (7973, 7)

Missing ratio:
Tg         93.6
FFV        11.8
Tc         90.8
Density    92.3
Rg         92.3
dtype: float64


In [6]:
ds1 = pd.read_csv(os.path.join(SUPP_DIR, "dataset1.csv"))  # Tc
ds2 = pd.read_csv(os.path.join(SUPP_DIR, "dataset2.csv"))  # SMILES only
ds3 = pd.read_csv(os.path.join(SUPP_DIR, "dataset3.csv"))  # Tg
ds4 = pd.read_csv(os.path.join(SUPP_DIR, "dataset4.csv"))  # FFV

print("Supplement shapes:")
print("dataset1:", ds1.shape)
print("dataset2:", ds2.shape)
print("dataset3:", ds3.shape)
print("dataset4:", ds4.shape)

Supplement shapes:
dataset1: (874, 2)
dataset2: (7208, 1)
dataset3: (46, 2)
dataset4: (862, 2)


In [7]:
main_train = train_df.dropna(subset=TARGETS, how="all").copy()
main_train["source"] = "train"
main_train = main_train[["id", "SMILES"] + TARGETS + ["source"]]

# dataset1: Tc
supp_tc = ds1.rename(columns={"TC_mean": "Tc"}).copy()
supp_tc["id"] = np.nan
supp_tc["Tg"] = np.nan
supp_tc["FFV"] = np.nan
supp_tc["Density"] = np.nan
supp_tc["Rg"] = np.nan
supp_tc["source"] = "supp_tc"
supp_tc = supp_tc[["id", "SMILES"] + TARGETS + ["source"]]

# dataset3: Tg
supp_tg = ds3.copy()
supp_tg["id"] = np.nan
supp_tg["FFV"] = np.nan
supp_tg["Tc"] = np.nan
supp_tg["Density"] = np.nan
supp_tg["Rg"] = np.nan
supp_tg["source"] = "supp_tg"
supp_tg = supp_tg[["id", "SMILES"] + TARGETS + ["source"]]

# dataset4: FFV
supp_ffv = ds4.copy()
supp_ffv["id"] = np.nan
supp_ffv["Tg"] = np.nan
supp_ffv["Tc"] = np.nan
supp_ffv["Density"] = np.nan
supp_ffv["Rg"] = np.nan
supp_ffv["source"] = "supp_ffv"
supp_ffv = supp_ffv[["id", "SMILES"] + TARGETS + ["source"]]

full_df = pd.concat([main_train, supp_tc, supp_tg, supp_ffv], ignore_index=True)

print("Full dataframe:", full_df.shape)
print(full_df["source"].value_counts())

print("\nLabel counts:")
for t in TARGETS:
    print(f"{t}: {full_df[t].notna().sum()}")

Full dataframe: (9755, 8)
source
train       7973
supp_tc      874
supp_ffv     862
supp_tg       46
Name: count, dtype: int64

Label counts:
Tg: 557
FFV: 7892
Tc: 1611
Density: 613
Rg: 614


In [8]:
def make_y_and_mask(row):
    y, mask = [], []
    for t in TARGETS:
        if pd.isna(row[t]):
            y.append(0.0)
            mask.append(0.0)
        else:
            y.append(float(row[t]))
            mask.append(1.0)
    return pd.Series([y, mask], index=["y", "mask"])

full_df[["y", "mask"]] = full_df.apply(make_y_and_mask, axis=1)
full_df["num_labels"] = full_df["mask"].apply(sum)

print(full_df["num_labels"].value_counts().sort_index())

gcn_df = full_df[["SMILES", "y", "mask", "source"]].copy()
print(gcn_df.head())

num_labels
1.0    9048
2.0     122
3.0     345
4.0     240
Name: count, dtype: int64
                                              SMILES  \
0                         *CC(*)c1ccccc1C(=O)OCCCCCC   
1  *Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...   
2  *Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...   
3  *Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...   
4  *Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...   

                                                 y                       mask  \
0  [0.0, 0.37464529, 0.2056666666666666, 0.0, 0.0]  [0.0, 1.0, 1.0, 0.0, 0.0]   
1                  [0.0, 0.3704102, 0.0, 0.0, 0.0]  [0.0, 1.0, 0.0, 0.0, 0.0]   
2                 [0.0, 0.37885964, 0.0, 0.0, 0.0]  [0.0, 1.0, 0.0, 0.0, 0.0]   
3                  [0.0, 0.3873239, 0.0, 0.0, 0.0]  [0.0, 1.0, 0.0, 0.0, 0.0]   
4                 [0.0, 0.35547017, 0.0, 0.0, 0.0]  [0.0, 1.0, 0.0, 0.0, 0.0]   

  source  
0  train  
1  train  
2  train  
3  train  
4  train  


In [9]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        int(atom.GetIsAromatic()),
    ]


def smiles_to_graph(smiles, y=None, mask=None, idx=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    x = torch.tensor(
        [atom_features(atom) for atom in mol.GetAtoms()],
        dtype=torch.float
    )

    n = x.size(0)
    adj = torch.zeros((n, n), dtype=torch.float)

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        adj[i, j] = 1.0
        adj[j, i] = 1.0

    # Add self-loops
    adj += torch.eye(n)

    # Symmetric normalization: D^{-1/2} A D^{-1/2}
    deg = adj.sum(dim=1)
    deg_inv_sqrt = torch.pow(deg, -0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    adj_norm = deg_inv_sqrt[:, None] * adj * deg_inv_sqrt[None, :]

    return {
        "x": x,
        "adj": adj_norm,
        "y": torch.tensor(y, dtype=torch.float) if y is not None else None,
        "mask": torch.tensor(mask, dtype=torch.float) if mask is not None else None,
        "idx": idx,
    }

In [10]:
graphs = []

for i in tqdm(range(len(gcn_df)), desc="Building graphs"):
    row = gcn_df.iloc[i]
    graph = smiles_to_graph(row["SMILES"], row["y"], row["mask"], idx=i)
    graphs.append(graph)

valid_indices = [i for i, g in enumerate(graphs) if g is not None]

print("Total rows:", len(graphs))
print("Valid graphs:", len(valid_indices))
print("Invalid graphs:", len(graphs) - len(valid_indices))

Building graphs: 100%|██████████| 9755/9755 [00:15<00:00, 611.95it/s]

Total rows: 9755
Valid graphs: 9755
Invalid graphs: 0


In [11]:
def collate_graphs(batch_graphs):
    batch_graphs = [g for g in batch_graphs if g is not None]

    max_nodes = max(g["x"].size(0) for g in batch_graphs)
    batch_size = len(batch_graphs)
    feat_dim = batch_graphs[0]["x"].size(1)

    x = torch.zeros((batch_size, max_nodes, feat_dim), dtype=torch.float)
    adj = torch.zeros((batch_size, max_nodes, max_nodes), dtype=torch.float)
    node_mask = torch.zeros((batch_size, max_nodes), dtype=torch.float)

    y = torch.zeros((batch_size, len(TARGETS)), dtype=torch.float)
    target_mask = torch.zeros((batch_size, len(TARGETS)), dtype=torch.float)
    idx = []

    for b, g in enumerate(batch_graphs):
        n = g["x"].size(0)
        x[b, :n] = g["x"]
        adj[b, :n, :n] = g["adj"]
        node_mask[b, :n] = 1.0

        y[b] = g["y"]
        target_mask[b] = g["mask"]
        idx.append(g["idx"])

    return {
        "x": x,
        "adj": adj,
        "node_mask": node_mask,
        "y": y,
        "target_mask": target_mask,
        "idx": torch.tensor(idx, dtype=torch.long),
    }


def make_batches(indices, shuffle=False):
    indices = list(indices)
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), BATCH_SIZE):
        batch_indices = indices[start:start + BATCH_SIZE]
        batch_graphs = [graphs[i] for i in batch_indices]
        yield collate_graphs(batch_graphs)

In [12]:
class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        # x: [B, N, F], adj: [B, N, N]
        h = torch.bmm(adj, x)
        h = self.linear(h)
        return h


class GCNRegressor(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=64, out_dim=5):
        super().__init__()

        self.gcn1 = DenseGCNLayer(in_dim, hidden_dim)
        self.gcn2 = DenseGCNLayer(hidden_dim, hidden_dim)

        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, batch):
        x = batch["x"]
        adj = batch["adj"]
        node_mask = batch["node_mask"]

        x = F.relu(self.gcn1(x, adj))
        x = F.relu(self.gcn2(x, adj))

        # Masked global mean pooling
        x = x * node_mask.unsqueeze(-1)
        graph_emb = x.sum(dim=1) / node_mask.sum(dim=1, keepdim=True).clamp(min=1.0)

        out = F.relu(self.fc1(graph_emb))
        out = self.fc2(out)

        return out

In [13]:
def move_batch_to_device(batch, device):
    return {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }


def masked_mae_loss(pred, target, mask):
    loss = torch.abs(pred - target) * mask
    return loss.sum() / mask.sum().clamp(min=1.0)


def train_one_epoch(model, train_idx, optimizer):
    model.train()
    losses = []

    for batch in make_batches(train_idx, shuffle=True):
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad()
        pred = model(batch)
        loss = masked_mae_loss(pred, batch["y"], batch["target_mask"])

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, val_idx):
    model.eval()
    losses = []

    for batch in make_batches(val_idx, shuffle=False):
        batch = move_batch_to_device(batch, device)

        pred = model(batch)
        loss = masked_mae_loss(pred, batch["y"], batch["target_mask"])
        losses.append(loss.item())

    return float(np.mean(losses))

In [14]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

n = len(gcn_df)

oof_pred = np.full((n, len(TARGETS)), np.nan, dtype=np.float32)
oof_true = np.full((n, len(TARGETS)), np.nan, dtype=np.float32)
oof_mask = np.zeros((n, len(TARGETS)), dtype=np.float32)

fold_results = []

for fold, (train_pos, val_pos) in enumerate(kf.split(valid_indices), start=1):
    print(f"\n========== Fold {fold} ==========")

    train_idx = [valid_indices[i] for i in train_pos]
    val_idx = [valid_indices[i] for i in val_pos]

    set_seed(SEED + fold)

    model = GCNRegressor(
        in_dim=3,
        hidden_dim=HIDDEN_DIM,
        out_dim=len(TARGETS)
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val = float("inf")
    best_state = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_idx, optimizer)
        val_loss = evaluate(model, val_idx)

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Fold {fold} | Epoch {epoch:02d} | "
            f"Train: {train_loss:.4f} | Val: {val_loss:.4f}"
        )

    model.load_state_dict(best_state)
    fold_results.append(best_val)

    print(f"Fold {fold} best val loss: {best_val:.4f}")

    model.eval()
    with torch.no_grad():
        for batch in make_batches(val_idx, shuffle=False):
            batch = move_batch_to_device(batch, device)

            pred = model(batch).cpu().numpy()
            true = batch["y"].cpu().numpy()
            mask = batch["target_mask"].cpu().numpy()
            idx = batch["idx"].cpu().numpy()

            oof_pred[idx] = pred
            oof_true[idx] = true
            oof_mask[idx] = mask

print("\nMean fold validation loss:", np.mean(fold_results))


========== Fold 1 ==========
Fold 1 | Epoch 01 | Train: 5.6242 | Val: 4.7737
Fold 1 | Epoch 02 | Train: 4.4872 | Val: 4.7064
Fold 1 | Epoch 03 | Train: 4.4681 | Val: 4.5880
Fold 1 | Epoch 04 | Train: 4.3863 | Val: 4.5580
Fold 1 | Epoch 05 | Train: 4.3457 | Val: 4.4942
Fold 1 | Epoch 06 | Train: 4.3510 | Val: 4.4644
Fold 1 | Epoch 07 | Train: 4.2660 | Val: 4.3823
Fold 1 | Epoch 08 | Train: 4.2516 | Val: 4.3743
Fold 1 | Epoch 09 | Train: 4.1769 | Val: 4.2510
Fold 1 | Epoch 10 | Train: 4.1050 | Val: 4.1418
Fold 1 | Epoch 11 | Train: 3.9910 | Val: 3.9855
Fold 1 | Epoch 12 | Train: 3.7962 | Val: 4.2210
Fold 1 | Epoch 13 | Train: 3.7233 | Val: 3.7613
Fold 1 | Epoch 14 | Train: 3.6058 | Val: 3.6674
Fold 1 | Epoch 15 | Train: 3.5540 | Val: 3.8347
Fold 1 | Epoch 16 | Train: 3.5308 | Val: 3.6083
Fold 1 | Epoch 17 | Train: 3.5243 | Val: 3.5984
Fold 1 | Epoch 18 | Train: 3.4835 | Val: 3.6551
Fold 1 | Epoch 19 | Train: 3.4852 | Val: 3.5816
Fold 1 | Epoch 20 | Train: 3.4551 | Val: 3.5662
Fold 1 bes

In [15]:
per_task_oof_mae = {}

for j, target in enumerate(TARGETS):
    valid = oof_mask[:, j] == 1
    mae = np.abs(oof_pred[valid, j] - oof_true[valid, j]).mean()
    per_task_oof_mae[target] = mae

overall_masked_oof_mae = np.abs(oof_pred - oof_true)[oof_mask == 1].mean()

print("\nPer-target OOF MAE:")
for target in TARGETS:
    print(f"{target}: {per_task_oof_mae[target]:.4f}")

print(f"\nOverall masked OOF MAE: {overall_masked_oof_mae:.4f}")
print("Fold val losses:", [round(x, 4) for x in fold_results])
print(f"Mean fold val loss: {np.mean(fold_results):.4f}")


Per-target OOF MAE:
Tg: 65.4833
FFV: 0.0370
Tc: 0.0735
Density: 0.1104
Rg: 3.1201

Overall masked OOF MAE: 3.4436
Fold val losses: [3.5662, 3.5485, 2.8224, 3.3779, 3.9163]
Mean fold val loss: 3.4463


In [16]:
def compute_oof_wmae(oof_pred, oof_true, oof_mask):
    n_i = []
    r_i = []

    for j in range(len(TARGETS)):
        valid = oof_mask[:, j] == 1
        yj = oof_true[valid, j]

        n_i.append(valid.sum())
        r_i.append(yj.max() - yj.min())

    n_i = np.array(n_i, dtype=np.float64)
    r_i = np.array(r_i, dtype=np.float64)

    K = len(TARGETS)
    sqrt_inv_n = np.sqrt(1.0 / n_i)
    norm_factor = K * sqrt_inv_n / sqrt_inv_n.sum()
    weights = (1.0 / r_i) * norm_factor

    total = 0.0
    count = 0

    for i in range(oof_pred.shape[0]):
        for j in range(len(TARGETS)):
            if oof_mask[i, j] == 1:
                total += weights[j] * abs(oof_pred[i, j] - oof_true[i, j])
                count += 1

    return total / count, weights, n_i, r_i


oof_wmae, weights, n_i, r_i = compute_oof_wmae(oof_pred, oof_true, oof_mask)

print(f"OOF weighted MAE: {oof_wmae:.6f}")
print("weights:", weights)
print("n_i:", n_i)
print("r_i:", r_i)

OOF weighted MAE: 0.044429
weights: [0.00214422 0.64231194 0.50667608 1.16067236 0.05078377]
n_i: [ 557. 7892. 1611.  613.  614.]
r_i: [6.20279724e+02 5.50104618e-01 1.54350007e+00 1.09230757e+00
 2.49445496e+01]


In [17]:
results_df = pd.DataFrame({
    "Target": TARGETS,
    "GCN_OOF_MAE": [per_task_oof_mae[t] for t in TARGETS],
    "n_labels": n_i.astype(int),
    "range": r_i,
    "weight": weights,
})

summary = {
    "overall_masked_oof_mae": float(overall_masked_oof_mae),
    "oof_weighted_mae": float(oof_wmae),
    "mean_fold_val_loss": float(np.mean(fold_results)),
    "fold_val_losses": [float(x) for x in fold_results],
}

display(results_df)

print("\n===== GCN OOF Summary =====")
for k, v in summary.items():
    print(k, ":", v)

,Target,GCN_OOF_MAE,n_labels,range,weight
0,Tg,65.483284,557,620.279724,0.002144
1,FFV,0.036976,7892,0.550105,0.642312
2,Tc,0.073500,1611,1.543500,0.506676
3,Density,0.110389,613,1.092308,1.160672
4,Rg,3.120125,614,24.944550,0.050784



===== GCN OOF Summary =====
overall_masked_oof_mae : 3.4435932636260986
oof_weighted_mae : 0.04442914272814512
mean_fold_val_loss : 3.4462873996403376
fold_val_losses : [3.566223910780715, 3.5485335763116352, 2.822400670712356, 3.3779408171406535, 3.916338023256327]


In [18]:
results_df.to_csv(os.path.join(OUTPUT_DIR, "gcn_per_target_oof_mae.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "gcn_oof_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Saved results to:", OUTPUT_DIR)

Saved results to: ./gcn_outputs


In [19]:
print("\n===== GCN OOF Results =====")
for target in TARGETS:
    print(f"{target}: {per_task_oof_mae[target]:.4f}")

print(f"\nOverall masked OOF MAE: {overall_masked_oof_mae:.4f}")
print(f"OOF weighted MAE: {oof_wmae:.4f}")
print(f"Mean fold validation loss: {np.mean(fold_results):.4f}")


===== GCN OOF Results =====
Tg: 65.4833
FFV: 0.0370
Tc: 0.0735
Density: 0.1104
Rg: 3.1201

Overall masked OOF MAE: 3.4436
OOF weighted MAE: 0.0444
Mean fold validation loss: 3.4463


In [20]:
# =========================
# TEST PREDICTION + SUBMISSION
# =========================

test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("Test shape:", test_df.shape)
print(sample_sub.head())

Test shape: (3, 2)
           id  Tg  FFV  Tc  Density  Rg
0  1109053969   0    0   0        0   0
1  1422188626   0    0   0        0   0
2  2032016830   0    0   0        0   0


In [21]:
# Build test graphs
test_graphs = []

for i in tqdm(range(len(test_df)), desc="Building test graphs"):
    row = test_df.iloc[i]
    
    graph = smiles_to_graph(
        row["SMILES"],
        y=[0.0] * len(TARGETS),
        mask=[0.0] * len(TARGETS),
        idx=i
    )
    
    test_graphs.append(graph)

print("Total test rows:", len(test_graphs))
print("Valid test graphs:", sum(g is not None for g in test_graphs))
print("Invalid test graphs:", sum(g is None for g in test_graphs))

Building test graphs: 100%|██████████| 3/3 [00:00<00:00, 345.08it/s]

Total test rows: 3
Valid test graphs: 3
Invalid test graphs: 0


In [22]:
def make_test_batches(test_graphs, batch_size=BATCH_SIZE):
    valid_graphs = [g for g in test_graphs if g is not None]
    
    for start in range(0, len(valid_graphs), batch_size):
        batch_graphs = valid_graphs[start:start + batch_size]
        yield collate_graphs(batch_graphs)

In [23]:
@torch.no_grad()
def predict_test(model, test_graphs):
    model.eval()
    
    preds = np.full((len(test_graphs), len(TARGETS)), np.nan, dtype=np.float32)
    
    for batch in make_test_batches(test_graphs):
        batch = move_batch_to_device(batch, device)
        pred = model(batch).cpu().numpy()
        idx = batch["idx"].cpu().numpy()
        
        preds[idx] = pred
    
    return preds

In [24]:
# Retrain 5-fold models and average test predictions
test_preds_all_folds = []

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (train_pos, val_pos) in enumerate(kf.split(valid_indices), start=1):
    print(f"\n========== Submission Fold {fold} ==========")

    train_idx = [valid_indices[i] for i in train_pos]
    val_idx = [valid_indices[i] for i in val_pos]

    set_seed(SEED + fold)

    model = GCNRegressor(
        in_dim=3,
        hidden_dim=HIDDEN_DIM,
        out_dim=len(TARGETS)
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val = float("inf")
    best_state = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_idx, optimizer)
        val_loss = evaluate(model, val_idx)

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Fold {fold} | Epoch {epoch:02d} | "
            f"Train: {train_loss:.4f} | Val: {val_loss:.4f}"
        )

    model.load_state_dict(best_state)
    print(f"Fold {fold} best val loss: {best_val:.4f}")

    fold_test_pred = predict_test(model, test_graphs)
    test_preds_all_folds.append(fold_test_pred)


========== Submission Fold 1 ==========
Fold 1 | Epoch 01 | Train: 5.6242 | Val: 4.7737
Fold 1 | Epoch 02 | Train: 4.4872 | Val: 4.7064
Fold 1 | Epoch 03 | Train: 4.4681 | Val: 4.5880
Fold 1 | Epoch 04 | Train: 4.3863 | Val: 4.5580
Fold 1 | Epoch 05 | Train: 4.3457 | Val: 4.4942
Fold 1 | Epoch 06 | Train: 4.3510 | Val: 4.4644
Fold 1 | Epoch 07 | Train: 4.2660 | Val: 4.3823
Fold 1 | Epoch 08 | Train: 4.2516 | Val: 4.3743
Fold 1 | Epoch 09 | Train: 4.1769 | Val: 4.2510
Fold 1 | Epoch 10 | Train: 4.1050 | Val: 4.1418
Fold 1 | Epoch 11 | Train: 3.9910 | Val: 3.9855
Fold 1 | Epoch 12 | Train: 3.7962 | Val: 4.2210
Fold 1 | Epoch 13 | Train: 3.7233 | Val: 3.7613
Fold 1 | Epoch 14 | Train: 3.6058 | Val: 3.6674
Fold 1 | Epoch 15 | Train: 3.5540 | Val: 3.8347
Fold 1 | Epoch 16 | Train: 3.5308 | Val: 3.6083
Fold 1 | Epoch 17 | Train: 3.5243 | Val: 3.5984
Fold 1 | Epoch 18 | Train: 3.4835 | Val: 3.6551
Fold 1 | Epoch 19 | Train: 3.4852 | Val: 3.5816
Fold 1 | Epoch 20 | Train: 3.4551 | Val: 3.5662

In [25]:
# Average predictions across folds
test_pred_mean = np.nanmean(np.stack(test_preds_all_folds, axis=0), axis=0)

print("Test prediction shape:", test_pred_mean.shape)
print(test_pred_mean[:5])

Test prediction shape: (3, 5)
[[143.08467      0.35607606   0.25414953   1.0552095   15.231651  ]
 [180.07579      0.3612744    0.24606447   1.0425406   17.672857  ]
 [130.85536      0.3558938    0.25061408   1.0444545   14.609654  ]]


In [26]:
# Create submission dataframe
submission = sample_sub.copy()

submission["Tg"] = test_pred_mean[:, 0]
submission["FFV"] = test_pred_mean[:, 1]
submission["Tc"] = test_pred_mean[:, 2]
submission["Density"] = test_pred_mean[:, 3]
submission["Rg"] = test_pred_mean[:, 4]

# Optional safety clipping for physically non-negative properties
submission["FFV"] = submission["FFV"].clip(lower=0)
submission["Tc"] = submission["Tc"].clip(lower=0)
submission["Density"] = submission["Density"].clip(lower=0)
submission["Rg"] = submission["Rg"].clip(lower=0)

print(submission.head())
print(submission.isna().sum())

           id          Tg       FFV        Tc   Density         Rg
0  1109053969  143.084671  0.356076  0.254150  1.055210  15.231651
1  1422188626  180.075790  0.361274  0.246064  1.042541  17.672857
2  2032016830  130.855362  0.355894  0.250614  1.044454  14.609654
id         0
Tg         0
FFV        0
Tc         0
Density    0
Rg         0
dtype: int64


In [27]:
# Save submission
submission_path = "/kaggle/working/submission.csv"

if not os.path.exists("/kaggle/working"):
    submission_path = "submission.csv"

submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print(submission.shape)

Saved submission to: /kaggle/working/submission.csv
(3, 6)
